# Module 4 Exercise (Solution): Multi-head self-attention and a full encoder block from scratch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nsteve2407/llm-transformers-course/blob/master/notebooks/04-attention/exercise_solution.ipynb)

Module page: [Module 4: Attention Is All You Need](https://nsteve2407.github.io/llm-transformers-course/modules/04-attention/)

`nn.MultiheadAttention`/`nn.TransformerEncoderLayer` are used only *after* the from-scratch implementation, as a correctness oracle -- never inside the implementation itself.

**Part A**: scaled dot-product attention, `softmax(QK^T/sqrt(d_k) + mask)V`, operating on `(batch, heads, seq, d_k)` tensors.

**Part B**: `MultiHeadAttention` module -- separate learned Q/K/V/output projections, reshape into heads, supports both self-attention and cross-attention.

**Part C**: positional encodings -- fixed sinusoidal (non-learned buffer) and learned (`nn.Embedding`), behind a flag.

**Part D**: position-wise FFN and a full encoder block (self-attn -> residual -> LayerNorm -> FFN -> residual -> LayerNorm), in both Post-LN and Pre-LN form, behind a flag.

**Part E**: causal (look-ahead) mask and padding mask construction, and combining both.

**Part F**: validate `MultiHeadAttention` against PyTorch's own `nn.MultiheadAttention` by copying weights and checking `torch.allclose` agreement.

**Part G**: validate the full encoder block (both LN placements) against `nn.TransformerEncoderLayer` the same way.

**Part H**: attention-weight heatmap for a toy sentence.

**Part I**: causal-mask verification -- a concrete, checkable proof (not just a comment) that future positions never leak into earlier ones.


In [ ]:
import os
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"SMOKE_TEST={SMOKE_TEST}, device={device}")


## Hyperparameters and design choices (documented judgment calls)

Everything in this notebook operates on synthetic tensors (no dataset download), so there is no
`SMOKE_TEST`-scaled data path -- the notebook is already small enough to run in well under a minute either
way. `SMOKE_TEST`/`device` above are kept only for structural consistency with the other module notebooks.

**Judgment calls made here:**
- `D_MODEL = 64`, `NUM_HEADS = 4` (so `d_k = d_model / num_heads = 16`), `SEQ_LEN = 16`, `BATCH = 4`,
  `D_FF = 4 * D_MODEL = 256` -- small enough to be fast and easy to inspect, matching the scale suggested by
  the exercise brief.
- **No dropout anywhere** in the from-scratch modules. This isn't a simplification for its own sake: Part F
  and Part G validate our modules against `nn.MultiheadAttention`/`nn.TransformerEncoderLayer` via exact
  weight-copying, and dropout is stochastic. Using `dropout=0.0` on the reference modules (their default
  attention dropout is already `0.0`; we pass `dropout=0.0` explicitly to `TransformerEncoderLayer`) makes
  the comparison deterministic and exact, rather than "close in distribution."
- `nn.LayerNorm` default `eps=1e-5` is used throughout, matching PyTorch's `TransformerEncoderLayer` default
  `layer_norm_eps`, so the Part G comparison isn't confounded by a numerical-stability constant mismatch.
- All linear projections (`W_q`, `W_k`, `W_v`, `W_o`, the FFN's two linears) use `nn.Linear`'s default
  `bias=True`, matching `nn.MultiheadAttention`'s default `bias=True` on `in_proj`/`out_proj`.


In [ ]:
D_MODEL = 64
NUM_HEADS = 4
assert D_MODEL % NUM_HEADS == 0, "d_model must be divisible by num_heads"
D_K = D_MODEL // NUM_HEADS
SEQ_LEN = 16
BATCH = 4
D_FF = 4 * D_MODEL

print(f"D_MODEL={D_MODEL}, NUM_HEADS={NUM_HEADS}, D_K={D_K}, SEQ_LEN={SEQ_LEN}, BATCH={BATCH}, D_FF={D_FF}")


## Part A: scaled dot-product attention

`softmax(QK^T / sqrt(d_k) + mask)V`, where `Q`, `K`, `V` have shape `(batch, heads, seq, d_k)` and `mask`
(if given) is additive -- `0` at positions that may be attended to, `-inf` at positions that may not -- and
broadcastable to `(batch, heads, seq_q, seq_k)`. We rely on `F.softmax`, which already subtracts the
per-row max internally before exponentiating, so this is numerically stable without any extra work here.


In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """Q, K, V: (batch, heads, seq_q_or_k, d_k). mask: additive, broadcastable to
    (batch, heads, seq_q, seq_k), containing 0 (attend) or -inf (do not attend).

    Returns (output, attn_weights) where output is (batch, heads, seq_q, d_k) and
    attn_weights is (batch, heads, seq_q, seq_k) (each row sums to 1).
    """
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)  # (batch, heads, seq_q, seq_k)
    if mask is not None:
        scores = scores + mask
    attn_weights = F.softmax(scores, dim=-1)  # F.softmax subtracts the row max internally -- stable.
    output = attn_weights @ V
    return output, attn_weights


In [ ]:
# Sanity check: shapes, and each attention-weight row sums to 1 (it's a distribution over keys).
Q_demo = torch.randn(BATCH, NUM_HEADS, SEQ_LEN, D_K)
K_demo = torch.randn(BATCH, NUM_HEADS, SEQ_LEN, D_K)
V_demo = torch.randn(BATCH, NUM_HEADS, SEQ_LEN, D_K)
out_demo, attn_demo = scaled_dot_product_attention(Q_demo, K_demo, V_demo)
print("output shape:", out_demo.shape, " attn_weights shape:", attn_demo.shape)
row_sums = attn_demo.sum(dim=-1)
assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-6), "attention rows must sum to 1"
print("attention rows sum to 1: OK")


## Part B: `MultiHeadAttention` module

Separate learned `d_model -> d_model` projections for Q, K, V, reshaped into `(batch, heads, seq, d_k)`,
run through `scaled_dot_product_attention`, reshaped back, and passed through an output projection. The
`forward` signature takes `query`, `key`, `value` separately (not a single `x`), so the same module handles
both **self-attention** (call with `query=key=value=x`) and **cross-attention** (call with `query` from one
sequence and `key`/`value` from another, e.g. decoder attending over an encoder's output) -- `key`/`value`
may even have a different sequence length than `query`.


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask=None):
        """query: (batch, seq_q, d_model). key, value: (batch, seq_k, d_model) (seq_k may
        differ from seq_q for cross-attention; for self-attention pass query=key=value=x).
        mask: additive, broadcastable to (batch, num_heads, seq_q, seq_k).

        Returns (output, attn_weights): output is (batch, seq_q, d_model), attn_weights is
        (batch, num_heads, seq_q, seq_k).
        """
        batch_size = query.size(0)
        seq_len_q = query.size(1)
        seq_len_k = key.size(1)

        Q = self.W_q(query)
        K = self.W_k(key)
        V = self.W_v(value)

        # (batch, seq, d_model) -> (batch, seq, heads, d_k) -> (batch, heads, seq, d_k)
        Q = Q.view(batch_size, seq_len_q, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)

        attn_out, attn_weights = scaled_dot_product_attention(Q, K, V, mask=mask)

        # (batch, heads, seq_q, d_k) -> (batch, seq_q, heads, d_k) -> (batch, seq_q, d_model)
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, seq_len_q, self.d_model)
        output = self.W_o(attn_out)
        return output, attn_weights


In [ ]:
# Sanity check: self-attention and cross-attention (different query/key-value seq lengths).
mha_demo = MultiHeadAttention(D_MODEL, NUM_HEADS)
x_demo = torch.randn(BATCH, SEQ_LEN, D_MODEL)
self_out, self_attn_w = mha_demo(x_demo, x_demo, x_demo)
print("self-attention output shape:", self_out.shape, " attn_weights shape:", self_attn_w.shape)
assert self_out.shape == (BATCH, SEQ_LEN, D_MODEL)
assert self_attn_w.shape == (BATCH, NUM_HEADS, SEQ_LEN, SEQ_LEN)

kv_demo = torch.randn(BATCH, SEQ_LEN + 3, D_MODEL)  # e.g. encoder output, longer than the query sequence
cross_out, cross_attn_w = mha_demo(x_demo, kv_demo, kv_demo)
print("cross-attention output shape:", cross_out.shape, " attn_weights shape:", cross_attn_w.shape)
assert cross_out.shape == (BATCH, SEQ_LEN, D_MODEL)
assert cross_attn_w.shape == (BATCH, NUM_HEADS, SEQ_LEN, SEQ_LEN + 3)
print("MultiHeadAttention self/cross-attention shape checks: OK")


## Part C: positional encodings -- sinusoidal and learned

`PositionalEncoding` exposes both behind a `mode` flag:
- `"sinusoidal"`: the fixed, non-learned `sin`/`cos` formula from the paper, computed once and stored as a
  buffer (`register_buffer`, so it moves with `.to(device)` and is saved in `state_dict()`, but has no
  gradient and is not a learned parameter).
- `"learned"`: an `nn.Embedding` over position indices `0..max_len-1`, trained like any other embedding
  table.

Both add the same-shaped `(1, seq_len, d_model)` positional signal to the input embeddings.


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, mode="sinusoidal"):
        super().__init__()
        assert mode in ("sinusoidal", "learned")
        self.mode = mode
        self.d_model = d_model

        if mode == "sinusoidal":
            position = torch.arange(max_len).unsqueeze(1).float()  # (max_len, 1)
            div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
            pe = torch.zeros(max_len, d_model)
            pe[:, 0::2] = torch.sin(position * div_term)
            pe[:, 1::2] = torch.cos(position * div_term)
            self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model), not a learned Parameter
        else:
            self.pe_embedding = nn.Embedding(max_len, d_model)

    def forward(self, x):
        """x: (batch, seq_len, d_model). Returns x + positional encoding, same shape."""
        batch_size, seq_len, _ = x.shape
        if self.mode == "sinusoidal":
            pe = self.pe[:, :seq_len, :]
        else:
            positions = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch_size, seq_len)
            pe = self.pe_embedding(positions)
        return x + pe


In [ ]:
# Compare the two PE flavors: same output shape, sinusoidal is a fixed non-learned buffer
# (no gradient, not in .parameters()), learned is a trainable nn.Embedding.
pe_sin = PositionalEncoding(D_MODEL, mode="sinusoidal")
pe_learned = PositionalEncoding(D_MODEL, mode="learned")
x_demo = torch.randn(BATCH, SEQ_LEN, D_MODEL)
out_sin = pe_sin(x_demo)
out_learned = pe_learned(x_demo)
print("sinusoidal PE output shape:", out_sin.shape, " #learned params:", sum(p.numel() for p in pe_sin.parameters()))
print("learned PE output shape:   ", out_learned.shape, " #learned params:", sum(p.numel() for p in pe_learned.parameters()))
assert out_sin.shape == out_learned.shape == x_demo.shape

# Visualize the fixed sinusoidal pattern itself (not the attention weights -- that's Part H).
plt.figure()
plt.imshow(pe_sin.pe[0, :SEQ_LEN, :].numpy(), cmap="RdBu", aspect="auto")
plt.xlabel("encoding dimension")
plt.ylabel("position")
plt.colorbar()
plt.title("Sinusoidal positional encoding (fixed, non-learned)")
plt.show()


## Part D: position-wise FFN and the encoder block (Post-LN and Pre-LN)

`PositionwiseFFN` is the standard two-layer `d_model -> d_ff -> d_model` MLP with a ReLU in between,
applied identically (with shared weights) at every sequence position.

`EncoderBlock` composes self-attention and the FFN into the two `LayerNorm`-wrapped sublayers, in either
of the two placements the field has used, selected by `ln_type`:
- **Post-LN** (the original *Attention Is All You Need* placement): `x = LN(x + Sublayer(x))` for each
  sublayer -- normalize *after* the residual add.
- **Pre-LN** (used by most modern large transformers, e.g. GPT-2 onward): `x = x + Sublayer(LN(x))` --
  normalize the sublayer's *input*, and add the (un-normalized) residual directly. Pre-LN is generally
  easier to train stably at depth without a warmup schedule, at some cost in final quality relative to a
  well-tuned Post-LN model.


In [ ]:
class PositionwiseFFN(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear2(F.relu(self.linear1(x)))


In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, ln_type="post"):
        super().__init__()
        assert ln_type in ("post", "pre")
        self.ln_type = ln_type
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = PositionwiseFFN(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        """x: (batch, seq, d_model). Returns (output, self_attn_weights)."""
        if self.ln_type == "post":
            attn_out, attn_weights = self.self_attn(x, x, x, mask)
            x = self.norm1(x + attn_out)
            ffn_out = self.ffn(x)
            x = self.norm2(x + ffn_out)
        else:  # pre-LN
            normed = self.norm1(x)
            attn_out, attn_weights = self.self_attn(normed, normed, normed, mask)
            x = x + attn_out
            ffn_out = self.ffn(self.norm2(x))
            x = x + ffn_out
        return x, attn_weights


In [ ]:
# Sanity check: both LN placements run and preserve shape.
x_demo = torch.randn(BATCH, SEQ_LEN, D_MODEL)
for ln_type in ("post", "pre"):
    block_demo = EncoderBlock(D_MODEL, NUM_HEADS, D_FF, ln_type=ln_type)
    out_demo, attn_w_demo = block_demo(x_demo)
    print(f"ln_type={ln_type:4s} output shape:", out_demo.shape, " attn_weights shape:", attn_w_demo.shape)
    assert out_demo.shape == x_demo.shape


## Part E: masking -- causal and padding masks

Two kinds of additive mask, both `0` (attend) / `-inf` (don't attend):
- **Causal / look-ahead mask**: position `i` may only attend to positions `j <= i` -- an upper-triangular
  `-inf` mask (strictly above the diagonal). Same for every batch element, shape `(seq_len, seq_len)`.
- **Padding mask**: built from each sequence's real (unpadded) length -- position `j` is masked for every
  query in that batch row if `j >= length`. Shape `(batch, seq_len)`, reshaped to `(batch, 1, 1, seq_len)` so
  it broadcasts across heads and query positions.

Combining both is just addition (both are `0`/`-inf`, so summing never produces `NaN` -- `-inf + -inf = -inf`
and `-inf + 0 = -inf`); the result correctly masks a key position if *either* mask says to.


In [ ]:
def build_causal_mask(seq_len, device=None):
    """Returns (seq_len, seq_len): 0 on/below the diagonal, -inf strictly above it."""
    return torch.triu(torch.full((seq_len, seq_len), float("-inf"), device=device), diagonal=1)


def build_padding_mask(lengths, max_len, device=None):
    """lengths: list/1D tensor of per-example valid lengths. Returns (batch, max_len): 0 for
    positions j < lengths[b], -inf for positions j >= lengths[b] (padding).
    """
    lengths_t = torch.as_tensor(lengths, device=device)
    positions = torch.arange(max_len, device=device).unsqueeze(0)  # (1, max_len)
    is_pad = positions >= lengths_t.unsqueeze(1)  # (batch, max_len)
    mask = torch.zeros(len(lengths), max_len, device=device)
    return mask.masked_fill(is_pad, float("-inf"))


In [ ]:
causal_mask = build_causal_mask(SEQ_LEN)
print("causal mask shape:", causal_mask.shape)
print("causal mask, top-left 5x5 corner (rows=query, cols=key):")
print(causal_mask[:5, :5])

lengths = [SEQ_LEN, SEQ_LEN - 4, SEQ_LEN - 7, SEQ_LEN]  # last two are padded
padding_mask = build_padding_mask(lengths, SEQ_LEN)
print("padding mask shape:", padding_mask.shape)

# Combine: (1, 1, seq, seq) + (batch, 1, 1, seq) -> broadcasts to (batch, 1, seq, seq).
combined_mask = causal_mask.unsqueeze(0).unsqueeze(0) + padding_mask.unsqueeze(1).unsqueeze(1)
print("combined mask shape:", combined_mask.shape)

# Verify: for every batch row b, key position j is masked at (b, :, i, j) iff j > i or j >= lengths[b].
for b, length in enumerate(lengths):
    for i in range(SEQ_LEN):
        for j in range(SEQ_LEN):
            expected_masked = (j > i) or (j >= length)
            actual_masked = bool(torch.isneginf(combined_mask[b, 0, i, j]))
            assert actual_masked == expected_masked, (b, i, j, expected_masked, actual_masked)
print("combined causal + padding mask matches the expected 0/-inf pattern for every (batch, query, key): OK")


## Part F: validate `MultiHeadAttention` against `nn.MultiheadAttention`

`nn.MultiheadAttention` packs the Q/K/V input projections into a single `in_proj_weight` of shape
`(3 * embed_dim, embed_dim)`, stacked `[W_q; W_k; W_v]` along dim 0 (plus a matching `in_proj_bias`), and
keeps the output projection as a separate `out_proj` (an `nn.Linear`). Both our `nn.Linear`-based `W_q`
(etc.) and PyTorch's `in_proj_weight` store weights in the same `(out_features, in_features)` convention
(`y = xW^T + b`), so copying is a direct `torch.cat` along dim 0 with **no extra transpose** -- the only
thing that requires care is getting the `[Q; K; V]` stacking order right.

We construct the reference with `batch_first=True` (PyTorch's default is seq-first) and `dropout=0.0` so
the comparison is an exact, deterministic `torch.allclose`, not merely "close in distribution."


In [ ]:
mha_custom = MultiHeadAttention(D_MODEL, NUM_HEADS)
mha_ref = nn.MultiheadAttention(embed_dim=D_MODEL, num_heads=NUM_HEADS, dropout=0.0, bias=True, batch_first=True)

with torch.no_grad():
    mha_ref.in_proj_weight.copy_(torch.cat([mha_custom.W_q.weight, mha_custom.W_k.weight, mha_custom.W_v.weight], dim=0))
    mha_ref.in_proj_bias.copy_(torch.cat([mha_custom.W_q.bias, mha_custom.W_k.bias, mha_custom.W_v.bias], dim=0))
    mha_ref.out_proj.weight.copy_(mha_custom.W_o.weight)
    mha_ref.out_proj.bias.copy_(mha_custom.W_o.bias)

mha_custom.eval()
mha_ref.eval()

x_val = torch.randn(BATCH, SEQ_LEN, D_MODEL)
with torch.no_grad():
    out_custom, _ = mha_custom(x_val, x_val, x_val)
    out_ref, _ = mha_ref(x_val, x_val, x_val, need_weights=False)

max_diff = (out_custom - out_ref).abs().max().item()
match = torch.allclose(out_custom, out_ref, atol=1e-5)
print(f"[no mask]     match={match}  max abs diff={max_diff:.3e}")
assert match, f"MultiHeadAttention does not match nn.MultiheadAttention (max abs diff={max_diff})"


In [ ]:
# Same comparison, now with the causal mask applied to both (attn_mask is nn.MultiheadAttention's
# name for the same additive-mask concept; a 2D (seq, seq) mask broadcasts over batch and heads).
with torch.no_grad():
    out_custom_m, _ = mha_custom(x_val, x_val, x_val, mask=causal_mask)
    out_ref_m, _ = mha_ref(x_val, x_val, x_val, attn_mask=causal_mask, need_weights=False)

max_diff_m = (out_custom_m - out_ref_m).abs().max().item()
match_m = torch.allclose(out_custom_m, out_ref_m, atol=1e-5)
print(f"[causal mask] match={match_m}  max abs diff={max_diff_m:.3e}")
assert match_m, f"Masked MultiHeadAttention does not match nn.MultiheadAttention (max abs diff={max_diff_m})"


## Part G: validate the full encoder block against `nn.TransformerEncoderLayer`

Same weight-copying idea, extended to the whole block: `self_attn.in_proj_weight`/`in_proj_bias`/`out_proj`
as above, plus `linear1`/`linear2` (the FFN) and `norm1`/`norm2` (the two LayerNorms) copied directly --
their parameter shapes and naming already match our own module's, since both use plain `nn.Linear`/
`nn.LayerNorm`. `nn.TransformerEncoderLayer`'s `norm_first` flag is exactly our `ln_type` distinction
(`norm_first=False` is Post-LN, `norm_first=True` is Pre-LN), so we validate both placements, not just
Post-LN.


In [ ]:
for ln_type in ("post", "pre"):
    torch.manual_seed(0)
    custom_block = EncoderBlock(D_MODEL, NUM_HEADS, D_FF, ln_type=ln_type)
    ref_layer = nn.TransformerEncoderLayer(
        d_model=D_MODEL, nhead=NUM_HEADS, dim_feedforward=D_FF, dropout=0.0,
        activation="relu", batch_first=True, norm_first=(ln_type == "pre"),
    )

    with torch.no_grad():
        sa = custom_block.self_attn
        ref_layer.self_attn.in_proj_weight.copy_(torch.cat([sa.W_q.weight, sa.W_k.weight, sa.W_v.weight], dim=0))
        ref_layer.self_attn.in_proj_bias.copy_(torch.cat([sa.W_q.bias, sa.W_k.bias, sa.W_v.bias], dim=0))
        ref_layer.self_attn.out_proj.weight.copy_(sa.W_o.weight)
        ref_layer.self_attn.out_proj.bias.copy_(sa.W_o.bias)
        ref_layer.linear1.weight.copy_(custom_block.ffn.linear1.weight)
        ref_layer.linear1.bias.copy_(custom_block.ffn.linear1.bias)
        ref_layer.linear2.weight.copy_(custom_block.ffn.linear2.weight)
        ref_layer.linear2.bias.copy_(custom_block.ffn.linear2.bias)
        ref_layer.norm1.weight.copy_(custom_block.norm1.weight)
        ref_layer.norm1.bias.copy_(custom_block.norm1.bias)
        ref_layer.norm2.weight.copy_(custom_block.norm2.weight)
        ref_layer.norm2.bias.copy_(custom_block.norm2.bias)

    custom_block.eval()
    ref_layer.eval()
    x_val = torch.randn(BATCH, SEQ_LEN, D_MODEL)
    with torch.no_grad():
        out_custom, _ = custom_block(x_val)
        out_ref = ref_layer(x_val)

    max_diff = (out_custom - out_ref).abs().max().item()
    match = torch.allclose(out_custom, out_ref, atol=1e-4)
    print(f"ln_type={ln_type:4s} match={match}  max abs diff={max_diff:.3e}")
    assert match, f"{ln_type}-LN EncoderBlock does not match nn.TransformerEncoderLayer (max abs diff={max_diff})"


## Part H: attention-weight visualization

A heatmap of one head's attention weights for a toy sentence. The module has random (untrained) weights,
so there's no claim of a meaningful learned pattern here -- the point is to see the mechanics: each row
(query position) is a probability distribution over columns (key positions), matching the row-sums-to-1
check from Part A.


In [ ]:
viz_tokens = ["the", "cat", "sat", "on", "the", "mat", "quietly", "."]
viz_vocab = {tok: i for i, tok in enumerate(sorted(set(viz_tokens)))}
viz_vocab_size = len(viz_vocab)

torch.manual_seed(1)
viz_embedding = nn.Embedding(viz_vocab_size, D_MODEL)
viz_pe = PositionalEncoding(D_MODEL, mode="sinusoidal")
viz_mha = MultiHeadAttention(D_MODEL, NUM_HEADS)
viz_mha.eval()

token_ids = torch.tensor([[viz_vocab[t] for t in viz_tokens]])  # (1, seq_len)
x_viz = viz_pe(viz_embedding(token_ids))
with torch.no_grad():
    _, viz_attn_weights = viz_mha(x_viz, x_viz, x_viz)  # (1, heads, seq_len, seq_len)

head_idx = 0
weights = viz_attn_weights[0, head_idx].numpy()
assert weights.shape == (len(viz_tokens), len(viz_tokens))

plt.figure()
plt.imshow(weights, cmap="viridis")
plt.xticks(range(len(viz_tokens)), viz_tokens, rotation=45, ha="right")
plt.yticks(range(len(viz_tokens)), viz_tokens)
plt.xlabel("key position")
plt.ylabel("query position")
plt.colorbar(label="attention weight")
plt.title(f"Self-attention weights, head {head_idx} (random-init weights)")
plt.tight_layout()
plt.show()


## Part I: causal-mask verification (concrete proof, not just visual)

Visual inspection of an attention heatmap isn't a proof. Here's a concrete, checkable one: take two inputs
that are identical up to some position `modify_pos`, and differ only at positions *after* it. Run the
encoder block on both **with the causal mask applied**. If the mask is correctly preventing future
positions from leaking into earlier ones, the outputs at positions `<= modify_pos` must be *exactly* equal
between the two runs (LayerNorm and the FFN are both per-position and don't mix across the sequence, so the
only way position `i`'s output could depend on position `j > i` is through masked-out attention weights) --
while the outputs at positions `> modify_pos` should (with overwhelming probability, since we resample with
fresh random floats) differ, confirming the test isn't vacuous.


In [ ]:
causal_block = EncoderBlock(D_MODEL, NUM_HEADS, D_FF, ln_type="post")
causal_block.eval()

modify_pos = SEQ_LEN // 2  # positions 0..modify_pos are shared; positions modify_pos+1.. differ
x1 = torch.randn(1, SEQ_LEN, D_MODEL)
x2 = x1.clone()
x2[:, modify_pos + 1:, :] = torch.randn_like(x2[:, modify_pos + 1:, :])

causal_mask_i = build_causal_mask(SEQ_LEN)
with torch.no_grad():
    out1, _ = causal_block(x1, mask=causal_mask_i)
    out2, _ = causal_block(x2, mask=causal_mask_i)

past_diff = (out1[:, :modify_pos + 1] - out2[:, :modify_pos + 1]).abs().max().item()
future_diff = (out1[:, modify_pos + 1:] - out2[:, modify_pos + 1:]).abs().max().item()
past_unaffected = torch.allclose(out1[:, :modify_pos + 1], out2[:, :modify_pos + 1], atol=1e-6)
future_differs = future_diff > 1e-4

print(f"positions <= {modify_pos} (must be unaffected by the future edit): max abs diff = {past_diff:.3e}")
print(f"positions >  {modify_pos} (do see the future edit):                max abs diff = {future_diff:.3e}")
assert past_unaffected, "causal mask is leaking: an earlier position's output changed when a later token was edited"
assert future_differs, "test is vacuous: the edited future positions should differ but don't"
print("causal mask verification: earlier positions are exactly unaffected by later edits, later positions do change. OK")


## Summary

Implemented and validated from scratch:
- **Scaled dot-product attention** (`softmax(QK^T/sqrt(d_k) + mask)V`) operating on `(batch, heads, seq,
  d_k)` tensors, with additive masking and stable softmax.
- **`MultiHeadAttention`**: separate learned Q/K/V/output projections, head reshape/concat, supporting both
  self-attention and cross-attention via its `(query, key, value)` call signature.
- **Positional encodings**: fixed sinusoidal (buffer, no gradient) and learned (`nn.Embedding`), behind a
  `mode` flag.
- **A full encoder block** (self-attn -> residual -> LayerNorm -> FFN -> residual -> LayerNorm) in both
  Post-LN and Pre-LN form, behind an `ln_type` flag.
- **Causal and padding masks**, built independently and combined by addition.
- **Numerical validation against PyTorch's own implementation**: weight-copied `MultiHeadAttention` matches
  `nn.MultiheadAttention` exactly (`torch.allclose`, with and without a causal mask), and the full encoder
  block matches `nn.TransformerEncoderLayer` exactly in both Post-LN (`norm_first=False`) and Pre-LN
  (`norm_first=True`) configurations.
- **Visualization** of a real attention-weight heatmap for a toy sentence.
- **A concrete causal-mask proof**: editing a future token provably leaves every earlier position's output
  bit-for-bit unchanged, while later positions do change -- not just "the heatmap looks triangular."
